In [1]:
pip install dash plotly scikit-learn pandas dash-bootstrap-components

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# -----------------------------
# Import Libraries
# -----------------------------
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, dcc, html
import dash_bootstrap_components as dbc


# -----------------------------
# Load Iris Dataset
# -----------------------------
iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=[
        'sepal_length',
        'sepal_width',
        'petal_length',
        'petal_width'
    ]
)

df['species'] = iris.target
df['species'] = df['species'].map({
    0:'Setosa',
    1:'Versicolor',
    2:'Virginica'
})


# -----------------------------
# KPI CALCULATIONS
# -----------------------------

# Total Records
total_records = len(df)

# Species Distribution
species_count = df['species'].value_counts().reset_index()
species_count.columns = ['species','count']

# Average Feature Values by Species
avg_features = df.groupby('species').mean().reset_index()

# Correlation Matrix
corr_matrix = df.drop(columns='species').corr()

# Outlier Detection (IQR Method)
outlier_summary = {}

for col in ['sepal_length','sepal_width','petal_length','petal_width']:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    outlier_summary[col] = len(outliers)

outlier_df = pd.DataFrame(
    list(outlier_summary.items()),
    columns=['Feature','Outlier Count']
)


# -----------------------------
# CHARTS
# -----------------------------

# Species Distribution Chart
fig_species = px.pie(
    species_count,
    names='species',
    values='count',
    title='Species Distribution'
)

# Boxplots
fig_box = px.box(
    df,
    x='species',
    y='petal_length',
    color='species',
    title='Petal Length Distribution by Species'
)

# Scatter Matrix
fig_scatter = px.scatter_matrix(
    df,
    dimensions=[
        'sepal_length',
        'sepal_width',
        'petal_length',
        'petal_width'
    ],
    color='species',
    title='Pair Plot (Scatter Matrix)'
)

# Correlation Heatmap
fig_heatmap = px.imshow(
    corr_matrix,
    text_auto=True,
    title="Feature Correlation Heatmap"
)


# -----------------------------
# DASH APPLICATION
# -----------------------------
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])


app.layout = dbc.Container([

    html.H1("Iris Dataset Dashboard", style={'textAlign':'center'}),

    html.Br(),

    # KPI Cards
    dbc.Row([

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H4("Total Records"),
                    html.H2(total_records)
                ])
            ])
        ),

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H4("Total Species"),
                    html.H2(df['species'].nunique())
                ])
            ])
        ),

    ]),

    html.Br(),

    # Charts
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_species))
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_box))
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_scatter))
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_heatmap))
    ])

])


# -----------------------------
# Run Server
# -----------------------------
if __name__ == '__main__':
    app.run(debug=True)

In [3]:
pip install dash plotly pandas dash-bootstrap-components

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
# -----------------------------
# Import Libraries
# -----------------------------
import pandas as pd
import numpy as np
import plotly.express as px

from dash import Dash, dcc, html
import dash_bootstrap_components as dbc

# -----------------------------
# Load Titanic Dataset
# -----------------------------
import seaborn as sns
df = sns.load_dataset('titanic')

# -----------------------------
# KPI CALCULATIONS
# -----------------------------

# Total passengers
total_passengers = len(df)

# Survival count
survived = df['survived'].sum()

# Survival rate
survival_rate = round((survived/total_passengers)*100,2)

# Average age
avg_age = round(df['age'].mean(),2)

# -----------------------------
# Charts
# -----------------------------

# Survival Distribution
fig_survival = px.pie(
    df,
    names='survived',
    title='Survival Distribution',
    color='survived'
)

# Passenger Class Distribution
fig_class = px.bar(
    df,
    x='pclass',
    title='Passenger Class Distribution'
)

# Age Distribution
fig_age = px.histogram(
    df,
    x='age',
    nbins=30,
    title='Age Distribution'
)

# Survival by Gender
fig_gender = px.bar(
    df,
    x='sex',
    color='survived',
    barmode='group',
    title='Survival by Gender'
)

# -----------------------------
# DASH APPLICATION
# -----------------------------
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([

    html.H1("Titanic Passenger Dashboard", style={'textAlign':'center'}),

    html.Br(),

    # KPI CARDS
    dbc.Row([

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H4("Total Passengers"),
                    html.H2(total_passengers)
                ])
            ])
        ),

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H4("Survived"),
                    html.H2(survived)
                ])
            ])
        ),

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H4("Survival Rate (%)"),
                    html.H2(survival_rate)
                ])
            ])
        ),

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    html.H4("Average Age"),
                    html.H2(avg_age)
                ])
            ])
        ),

    ]),

    html.Br(),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_survival))
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_class))
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_age))
    ]),

    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_gender))
    ])

])

# -----------------------------
# Run Server
# -----------------------------
if __name__ == '__main__':
    app.run(debug=True)